# NB34 — Genus-level MWAS: CLR abundance vs USGS soil metals

For each genus × metal × control combination: OLS regression of CLR-transformed  
genus abundance against log1p metal concentration across MicrobeAtlas USA soil samples.  

**Prerequisite:** NB33 must have run first (needs `nb33_sample_master.parquet`).  

**CLR transform:** centred log-ratio = log(count + 0.5) − mean_g[log(count_g + 0.5)]  
per sample. Handles compositionality; pseudocount 0.5 handles zeros.  

**Output:** `nb34_genus_mwas_results.csv` — (genus, metal, ctrl, beta, p, FDR),  
joined to nb25 KO presence for annotation.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H, grid_h
apply_style()

ROOT   = Path('/home/hmacgregor/BERIL-research-observatory')
CME    = ROOT / 'projects/comprehensive_metal_ecology'
USAENV = ROOT / 'projects/usa_env_bioindicators'
DATA   = CME / 'data'
FIGS   = CME / 'figures'

NB33_MASTER   = DATA / 'nb33_sample_master.parquet'
NB25_KO_PATH  = DATA / 'nb25_ko_presence_matrix.parquet'
OTU_PATH      = USAENV / 'data/nb02_otu_long_cache.parquet'
CURATED_PATH  = DATA / 'curated_mrg_ko_ids_v2.csv'
OUT_PATH      = DATA / 'nb34_genus_mwas_results.csv'
OUT_KO_PATH   = DATA / 'nb34_genus_mwas_ko_annotated.csv'

assert NB33_MASTER.exists(), f'Run NB33 first — {NB33_MASTER} not found'
assert OTU_PATH.exists(),    f'OTU cache not found: {OTU_PATH}'
print('Setup done.')

Setup done.


In [2]:
# ── Load NB33 sample master ─────────────────────────────────────────────
print('Loading NB33 sample master...')
samp_env = pd.read_parquet(NB33_MASTER)
print(f'  Sample master: {samp_env.shape}')
print(f'  Columns: {list(samp_env.columns)}')

# Metal columns have usgs_ prefix (e.g. usgs_ag, usgs_as, usgs_au ...)
# NON_METAL excludes env controls, coords, and distance column
NON_METAL = {'sample_id', 'lat', 'lon', 'lat_abs', 'soil_ph', 'soil_soc',
             'wtd_m', 'wc_mat', 'wc_map', 'p_oxic', 'usgs_dist_deg'}
METAL_COLS = sorted([c for c in samp_env.columns
                     if c not in NON_METAL
                     and samp_env[c].dtype in [np.float64, np.float32, float]
                     and samp_env[c].notna().sum() > 1000])
# Strip usgs_ prefix for display and cross-notebook matching
METALS = [c.replace('usgs_', '') for c in METAL_COLS]
print(f'\nMetal columns ({len(METAL_COLS)}): {METALS[:8]} ...')

# Log1p transform metals in sample master
for mc in METAL_COLS:
    samp_env[f'{mc}_log'] = np.log1p(samp_env[mc])
METAL_LOG_COLS = [f'{mc}_log' for mc in METAL_COLS]

# Coverage check
for mc, m in zip(METAL_COLS[:5], METALS[:5]):
    print(f'  {m}: {samp_env[mc].notna().sum():,} samples')

Loading NB33 sample master...
  Sample master: (6034, 60)
  Columns: ['sample_id', 'lat', 'lon', 'usgs_ag', 'usgs_as', 'usgs_au', 'usgs_b', 'usgs_ba', 'usgs_be', 'usgs_bi', 'usgs_cd', 'usgs_ce', 'usgs_co', 'usgs_cr', 'usgs_cs', 'usgs_cu', 'usgs_eu', 'usgs_ga', 'usgs_ge', 'usgs_hf', 'usgs_hg', 'usgs_in', 'usgs_la', 'usgs_li', 'usgs_lu', 'usgs_mo', 'usgs_nb', 'usgs_nd', 'usgs_ni', 'usgs_pb', 'usgs_pd', 'usgs_pt', 'usgs_rb', 'usgs_re', 'usgs_sb', 'usgs_sc', 'usgs_se', 'usgs_sm', 'usgs_sn', 'usgs_sr', 'usgs_ta', 'usgs_tb', 'usgs_te', 'usgs_th', 'usgs_tl', 'usgs_u', 'usgs_v', 'usgs_w', 'usgs_y', 'usgs_yb', 'usgs_zn', 'usgs_zr', 'usgs_dist_deg', 'soil_ph', 'soil_soc', 'wtd_m', 'wc_mat', 'wc_map', 'p_oxic', 'lat_abs']

Metal columns (48): ['ag', 'as', 'au', 'b', 'ba', 'be', 'bi', 'cd'] ...
  ag: 1,309 samples
  as: 6,010 samples
  au: 1,936 samples
  b: 1,983 samples
  ba: 6,010 samples


In [3]:
# ── Metal PC1 — geochemical background control ───────────────────────────
# Reproduce from NB33: PC1 captures the shared lithology axis
# (Ga/Zn/As/Nd/Ba/Sc/U/V/Sm/Tb top loadings, ~48% variance explained).
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

log_notna = samp_env[METAL_LOG_COLS].notna().mean()
pca_metal_cols = [c for c in METAL_LOG_COLS if log_notna[c] >= 0.30]
print(f'Metals entering PCA: {len(pca_metal_cols)} (≥30% sample coverage)')

pca_mask = samp_env[pca_metal_cols].notna().all(axis=1)
X_pca = samp_env.loc[pca_mask, pca_metal_cols].values
scaler = StandardScaler()
pca_obj = PCA(n_components=3)
pca_obj.fit(scaler.fit_transform(X_pca))
print(f'PC1 explains {pca_obj.explained_variance_ratio_[0]*100:.1f}% of metal variance')

X_all = scaler.transform(samp_env[pca_metal_cols].fillna(samp_env[pca_metal_cols].mean()).values)
pc_scores = pca_obj.transform(X_all)
samp_env['metal_pc1'] = np.where(pca_mask, pc_scores[:, 0], np.nan)

valid_pc1 = samp_env['metal_pc1'].notna()
samp_env['metal_pc1_z'] = np.nan
samp_env.loc[valid_pc1, 'metal_pc1_z'] = stats.zscore(samp_env.loc[valid_pc1, 'metal_pc1'])
print(f'Samples with metal PC1: {valid_pc1.sum():,}')

Metals entering PCA: 40 (≥30% sample coverage)
PC1 explains 89.8% of metal variance
Samples with metal PC1: 1,648


In [4]:
# ── Load MicrobeAtlas USA genus counts + CLR transform ───────────────────
print('Loading OTU long format...')
otu = pd.read_parquet(OTU_PATH)
print(f'  OTU: {otu.shape}, {otu.sample_id.nunique():,} samples, {otu.genus_lower.nunique():,} genera')

# Keep only samples present in sample master
valid_sids = set(samp_env['sample_id'])
otu = otu[otu['sample_id'].isin(valid_sids)].copy()
print(f'  After filtering to NB33 samples: {otu.shape}')

# CLR transform per sample
# CLR_g(s) = log(count_g(s) + 0.5) − mean_{g'} [log(count_{g'}(s) + 0.5)]
# Handles zeros via pseudocount; handles compositionality.
print('Computing CLR per sample...')
otu['log_pseudo'] = np.log(otu['count'] + 0.5)
sample_log_mean = otu.groupby('sample_id')['log_pseudo'].transform('mean')
otu['clr'] = otu['log_pseudo'] - sample_log_mean

# Minimum genus detection filter
MIN_GENUS_SAMPLES = 100  # genus must appear in ≥100 samples (count > 0)
genus_det = otu[otu['count'] > 0].groupby('genus_lower')['sample_id'].nunique()
KEEP_GENERA = genus_det[genus_det >= MIN_GENUS_SAMPLES].index.tolist()
print(f'Genera detected in ≥{MIN_GENUS_SAMPLES} samples: {len(KEEP_GENERA):,}')

otu = otu[otu['genus_lower'].isin(KEEP_GENERA)].copy()
print(f'  OTU after genus filter: {otu.shape}')

Loading OTU long format...


  OTU: (346716, 8), 6,034 samples, 200 genera
  After filtering to NB33 samples: (346716, 8)
Computing CLR per sample...
Genera detected in ≥100 samples: 200
  OTU after genus filter: (346716, 10)


In [5]:
# ── Merge environmental data + standardise controls ─────────────────────
print('Merging env data onto OTU...')
ENV_COLS = METAL_LOG_COLS + ['soil_ph', 'soil_soc', 'wtd_m', 'wc_mat', 'p_oxic', 'metal_pc1_z']
otu_env = otu[['sample_id', 'genus_lower', 'clr']].merge(
    samp_env[['sample_id'] + ENV_COLS].copy(),
    on='sample_id', how='inner'
)
print(f'  OTU × env: {otu_env.shape}')

# Z-score env controls across all samples (for comparability)
CTRL_RAW = ['soil_ph', 'soil_soc', 'wtd_m', 'wc_mat', 'p_oxic']
for col in CTRL_RAW:
    valid = otu_env[col].notna()
    otu_env[f'{col}_z'] = np.nan
    if valid.sum() > 10:
        otu_env.loc[valid, f'{col}_z'] = stats.zscore(otu_env.loc[valid, col])

# Already have metal_pc1_z from sample master

CTRL_COMBOS = [
    ('none',                              []),
    ('pH',                                ['soil_ph_z']),
    ('redox',                             ['p_oxic_z']),
    ('pH+redox',                          ['soil_ph_z', 'p_oxic_z']),
    ('pH+SOC+WTD',                        ['soil_ph_z', 'soil_soc_z', 'wtd_m_z']),
    ('pH+SOC+WTD+redox',                  ['soil_ph_z', 'soil_soc_z', 'wtd_m_z', 'p_oxic_z']),
    ('metalPC1',                          ['metal_pc1_z']),
    ('pH+SOC+WTD+redox+metalPC1',         ['soil_ph_z', 'soil_soc_z', 'wtd_m_z', 'p_oxic_z', 'metal_pc1_z']),
    ('pH+SOC+WTD+redox+metalPC1+MAT',     ['soil_ph_z', 'soil_soc_z', 'wtd_m_z', 'p_oxic_z', 'metal_pc1_z', 'wc_mat_z']),
]

# Check which controls are actually available
avail_ctrl_z = [f'{c}_z' for c in CTRL_RAW] + ['metal_pc1_z']
avail_ctrl_z = [c for c in avail_ctrl_z if c in otu_env.columns and otu_env[c].notna().sum() > 1000]
print(f'Available controls: {avail_ctrl_z}')

CTRL_COMBOS_FINAL = []
seen = set()
for name, cols in CTRL_COMBOS:
    filtered = [c for c in cols if c in avail_ctrl_z]
    key = tuple(sorted(filtered))
    if key not in seen:
        seen.add(key)
        CTRL_COMBOS_FINAL.append((name, filtered))
print(f'Control combos after dedup: {[c[0] for c in CTRL_COMBOS_FINAL]}')

Merging env data onto OTU...


  OTU × env: (346716, 57)
Available controls: ['soil_ph_z', 'soil_soc_z', 'wtd_m_z', 'wc_mat_z', 'p_oxic_z', 'metal_pc1_z']
Control combos after dedup: ['none', 'pH', 'redox', 'pH+redox', 'pH+SOC+WTD', 'pH+SOC+WTD+redox', 'metalPC1', 'pH+SOC+WTD+redox+metalPC1', 'pH+SOC+WTD+redox+metalPC1+MAT']


In [6]:
# ── Genus-level MWAS: CLR(genus, s) ~ metal_z + controls ───────────────
# For each genus × metal × ctrl: OLS across samples.
# Predictor = Z-scored metal_log (standardised within subset) so that beta
# is in units of ΔCLR per 1-SD change in log(metal) — comparable across elements.
# FDR is applied per (metal, ctrl) family after the loop.

MIN_N = 50  # minimum samples per regression

n_max = len(KEEP_GENERA) * len(METAL_COLS) * len(CTRL_COMBOS_FINAL)
print(f'Genus MWAS: {len(KEEP_GENERA):,} genera × {len(METAL_COLS)} metals × '
      f'{len(CTRL_COMBOS_FINAL)} combos = {n_max:,} max fits')

genus_mwas_rows = []

for gi, genus in enumerate(KEEP_GENERA):
    g_data = otu_env[otu_env['genus_lower'] == genus].copy()
    if len(g_data) < MIN_N:
        continue

    # metal_col = 'usgs_ag_log'; metal = 'ag' (stripped, for results)
    for metal_col_log, metal in zip(METAL_LOG_COLS, METALS):
        for ctrl_name, ctrl_cols in CTRL_COMBOS_FINAL:
            required = [metal_col_log, 'clr'] + ctrl_cols
            mask = g_data[required].notna().all(axis=1)
            sub = g_data.loc[mask]
            if len(sub) < MIN_N:
                continue

            y = sub['clr'].values

            # Z-score metal within this subset so betas are comparable across elements
            metal_vals = sub[metal_col_log].values
            metal_std = metal_vals.std()
            if metal_std < 1e-8:  # near-constant in this subset (e.g. trace element in uniform geology)
                continue
            metal_z_vals = (metal_vals - metal_vals.mean()) / metal_std

            X = np.column_stack(
                [np.ones(len(sub)),
                 metal_z_vals]
                + [sub[c].values for c in ctrl_cols]
            )
            try:
                betas, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
                yhat   = X @ betas
                rss    = np.sum((y - yhat) ** 2)
                tss    = np.sum((y - y.mean()) ** 2)
                df_r   = len(sub) - X.shape[1]
                sigma2 = rss / df_r if df_r > 0 else np.nan
                XtXinv = np.linalg.inv(X.T @ X)
                se_all = np.sqrt(np.maximum(np.diag(XtXinv) * sigma2, 0))
                beta_metal = betas[1]
                se_metal   = se_all[1]
                tstat = beta_metal / se_metal if se_metal > 0 else np.nan
                pval  = 2 * stats.t.sf(abs(tstat), df=df_r) if np.isfinite(tstat) else np.nan
                r2    = 1 - rss / tss if tss > 0 else np.nan
            except Exception:
                continue

            genus_mwas_rows.append({
                'genus':  genus,
                'metal':  metal,   # stripped name (e.g. 'ag'), matches NB33 PGLS
                'ctrl':   ctrl_name,
                'n':      len(sub),
                'beta':   beta_metal,   # ΔCLR per 1-SD increase in log1p(metal ppm)
                'se':     se_metal,
                'tstat':  tstat,
                'pval':   pval,
                'r2':     r2,
            })

    if (gi + 1) % 100 == 0 or gi == 0:
        print(f'  Genus {gi+1}/{len(KEEP_GENERA)}: {genus}, rows so far: {len(genus_mwas_rows):,}')

genus_mwas = pd.DataFrame(genus_mwas_rows)
print(f'\nFits complete: {len(genus_mwas):,} rows')

Genus MWAS: 200 genera × 48 metals × 9 combos = 86,400 max fits


  Genus 1/200: acanthamoeba, rows so far: 413


  Genus 100/200: microvirga, rows so far: 38,608


  Genus 200/200: williamsia, rows so far: 76,107

Fits complete: 76,107 rows


In [7]:
# ── FDR correction per (metal, ctrl) family ─────────────────────────────
fdr_chunks = []
for (metal, ctrl), grp in genus_mwas.groupby(['metal', 'ctrl']):
    grp = grp.copy()
    valid = grp['pval'].notna()
    grp['pval_fdr'] = np.nan
    grp['sig_fdr']  = False
    if valid.sum() > 1:
        reject, padj, _, _ = multipletests(grp.loc[valid, 'pval'].values, method='fdr_bh')
        grp.loc[valid, 'pval_fdr'] = padj
        grp.loc[valid, 'sig_fdr']  = reject
    fdr_chunks.append(grp)

genus_mwas = pd.concat(fdr_chunks, ignore_index=True)
n_sig = int(genus_mwas['sig_fdr'].sum())
print(f'Total FDR-significant genus × metal hits: {n_sig:,}')

# Summary per ctrl
sig_by_ctrl = genus_mwas.groupby('ctrl')['sig_fdr'].sum().sort_values(ascending=False)
print('\nSignificant hits by control combo:')
print(sig_by_ctrl.to_string())

# Summary per metal (kitchen-sink control)
kitchen = 'pH+SOC+WTD+redox+metalPC1'
if kitchen not in genus_mwas['ctrl'].values:
    kitchen = genus_mwas['ctrl'].value_counts().index[-1]  # most-controlled available
sig_by_metal = (genus_mwas[genus_mwas['ctrl'] == kitchen]
                .groupby('metal')['sig_fdr'].sum().sort_values(ascending=False))
print(f'\nSignificant genera per metal ({kitchen} control):')
print(sig_by_metal[sig_by_metal > 0].to_string())

genus_mwas.to_csv(OUT_PATH, index=False)
print(f'\nSaved raw results → {OUT_PATH.name}')

Total FDR-significant genus × metal hits: 35,987

Significant hits by control combo:
ctrl
none                             6582
redox                            6102
pH                               5929
pH+redox                         5509
pH+SOC+WTD                       5452
pH+SOC+WTD+redox                 4804
metalPC1                          714
pH+SOC+WTD+redox+metalPC1         450
pH+SOC+WTD+redox+metalPC1+MAT     445

Significant genera per metal (pH+SOC+WTD+redox+metalPC1 control):
metal
pd    25
pt    25
b     10
ba    10
be    10
bi    10
cd    10
ce    10
as    10
au    10
cr    10
co    10
cs    10
cu    10
ge    10
hg    10
eu    10
ga    10
la    10
in    10
li    10
mo    10
nd    10
nb    10
pb    10
ni    10
rb    10
sb    10
sc    10
se    10
sn    10
sr    10
te    10
th    10
tl    10
u     10
v     10
w     10
y     10
yb    10
zn    10
zr    10



Saved raw results → nb34_genus_mwas_results.csv


In [8]:
# ── Annotate significant hits with KOs from nb25 ─────────────────────────
print('Loading nb25 KO presence matrix...')
nb25 = pd.read_parquet(NB25_KO_PATH)
nb25['genus_lower'] = nb25['genus_lower'].str.replace('g__', '', regex=False)

# Load curated MHG KO list for labelling
curated = pd.read_csv(CURATED_PATH)
ko_meta = curated[['KO', 'gene_name', 'primary_category']].drop_duplicates('KO').copy()

# nb25 only has (genus_lower, ko, n_genomes_with_ko); get n_genomes from density CSV
spark_df = pd.read_csv(DATA / '01_genus_ko_density_spark.csv')
nb25 = nb25.merge(spark_df[['genus_lower', 'n_genomes']], on='genus_lower', how='inner')
nb25 = nb25[nb25['n_genomes'] >= 2].copy()
nb25['pres_frac'] = nb25['n_genomes_with_ko'] / nb25['n_genomes'].clip(lower=1)
print(f'nb25: {len(nb25):,} KO–genus rows ({nb25.genus_lower.nunique():,} genera)')

# For each significant (genus, metal, ctrl) hit, attach the KOs that genus carries
sig = genus_mwas[genus_mwas['sig_fdr']].copy()
print(f'Significant hits to annotate: {len(sig):,}')

# Join: sig × nb25 on genus_lower
sig_ko = sig.merge(
    nb25[['genus_lower', 'ko', 'pres_frac', 'n_genomes']],
    left_on='genus', right_on='genus_lower', how='left'
).drop(columns='genus_lower')

# Attach KO metadata (gene_name, category)
sig_ko = sig_ko.merge(ko_meta.rename(columns={'KO': 'ko'}), on='ko', how='left')
print(f'Sig hits × KO rows: {len(sig_ko):,}')

sig_ko.to_csv(OUT_KO_PATH, index=False)
print(f'Saved KO-annotated results → {OUT_KO_PATH.name}')

# Top genus × metal hits (kitchen-sink control), ordered by p_fdr
print(f'\nTop 20 hits ({kitchen} control):')
top = (genus_mwas[genus_mwas['ctrl'] == kitchen]
       .sort_values('pval_fdr')
       .head(20)
       [['genus', 'metal', 'n', 'beta', 'se', 'pval', 'pval_fdr', 'r2']])
print(top.to_string(index=False))

Loading nb25 KO presence matrix...
nb25: 416,414 KO–genus rows (8,255 genera)
Significant hits to annotate: 35,987


Sig hits × KO rows: 2,446,727


Saved KO-annotated results → nb34_genus_mwas_ko_annotated.csv

Top 20 hits (pH+SOC+WTD+redox+metalPC1 control):
          genus metal   n       beta        se         pval     pval_fdr       r2
     lysobacter    pt 278  -0.406280  0.059398 5.252983e-11 7.984534e-09 0.160440
     lysobacter    pd 278  -0.400904  0.058612 5.252983e-11 7.984534e-09 0.160440
flavisolibacter    pt 278  -0.337923  0.058987 2.685233e-08 2.040777e-06 0.194832
flavisolibacter    pd 278  -0.331422  0.057852 2.685233e-08 2.040777e-06 0.194832
flavisolibacter    cd 348  -3.314992  0.611849 1.141235e-07 1.540668e-05 0.539089
flavisolibacter    bi 348  42.279519  7.803547 1.141235e-07 1.597730e-05 0.539089
flavisolibacter    sb 348 -86.929621 16.044632 1.141235e-07 1.609142e-05 0.539089
flavisolibacter     w 348 -78.888043 14.560395 1.141235e-07 1.620554e-05 0.539089
flavisolibacter    ge 348 -12.725339  2.348721 1.141235e-07 1.620554e-05 0.539089
flavisolibacter    ce 348 -26.019147  4.802364 1.141235e-07 1.631967

In [9]:
# ── KO-level summary: which KOs appear in the most metal-significant genera?
# For each (KO, metal, ctrl): count how many significant genera carry that KO,
# and the mean absolute beta weighted by pres_frac.

PRES_FRAC_MIN = 0.50  # KO must be in ≥50% of genomes in a genus to count
CTRL_REPORT   = kitchen  # kitchen-sink control for main report

# Filter to curated Tier 1/2 KOs only for the main summary
tier12_kos = curated[curated['evidence_tier'].isin(['Tier 1', 'Tier 2'])]['KO'].tolist()

sig_ko_strict = sig_ko[
    (sig_ko['ctrl']      == CTRL_REPORT) &
    (sig_ko['pres_frac'] >= PRES_FRAC_MIN) &
    (sig_ko['ko'].isin(tier12_kos))
].copy()
print(f'Sig hits with Tier 1/2 KOs (pres_frac≥{PRES_FRAC_MIN}, ctrl={CTRL_REPORT}): {len(sig_ko_strict):,}')

ko_summary = (sig_ko_strict
    .groupby(['ko', 'gene_name', 'primary_category', 'metal'])
    .agg(
        n_sig_genera=('genus',  'nunique'),
        mean_abs_beta=('beta',  lambda x: x.abs().mean()),
        min_pval_fdr=('pval_fdr', 'min'),
    )
    .reset_index()
    .sort_values(['metal', 'n_sig_genera'], ascending=[True, False])
)

print(f'\nKO-level summary ({CTRL_REPORT} control, pres_frac≥{PRES_FRAC_MIN}):')
print(ko_summary.head(30).to_string(index=False))

ko_summary.to_csv(DATA / 'nb34_ko_metal_summary.csv', index=False)
print('Saved → nb34_ko_metal_summary.csv')

Sig hits with Tier 1/2 KOs (pres_frac≥0.5, ctrl=pH+SOC+WTD+redox+metalPC1): 810

KO-level summary (pH+SOC+WTD+redox+metalPC1 control, pres_frac≥0.5):
    ko gene_name          primary_category metal  n_sig_genera  mean_abs_beta  min_pval_fdr
K03446      emrB Resistance/Detoxification    as             2       7.808904      0.000048
K15726      czcA Resistance/Detoxification    as             2       5.863473      0.001792
K17686      copA Resistance/Detoxification    as             2       6.349664      0.000048
K02007      cbiM     Transport/Homeostasis    as             1       8.458486      0.000048
K02009      cbiN     Transport/Homeostasis    as             1       8.458486      0.000048
K02012      afuA     Transport/Homeostasis    as             1       8.458486      0.000048
K02225     cobC1     Cofactor Biosynthesis    as             1       8.458486      0.000048
K02230      cobN     Transport/Homeostasis    as             1       8.458486      0.000048
K03325      ACR3 Resis

In [10]:
# ── Figure 1: N significant genera per metal × control (heatmap) ─────────
from figure_style import annotate_n

pivot = (genus_mwas[genus_mwas['sig_fdr']]
         .groupby(['metal', 'ctrl'])['genus']
         .nunique()
         .unstack(fill_value=0))

ctrl_order = [c[0] for c in CTRL_COMBOS_FINAL if c[0] in pivot.columns]
pivot = pivot.reindex(columns=ctrl_order, fill_value=0)

fig, ax = plt.subplots(figsize=(FIGW['full'], max(ROW_H, len(pivot) * 0.22)))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', vmin=0)
ax.set_xticks(range(len(ctrl_order)))
ax.set_xticklabels(ctrl_order, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=7)
plt.colorbar(im, ax=ax, label='N significant genera (FDR<5%)', shrink=0.8)
ax.set_xlabel('Control combination')
ax.set_ylabel('USGS element')
ax.set_title('Genus-level MWAS: significant genera per metal × control')
fig.suptitle('NB34 — CLR genus abundance vs USGS soil metals', y=1.02)
save(fig, FIGS / 'nb34_genus_mwas_heatmap')
print('Figure 1 saved.')

Figure 1 saved.


In [11]:
# ── Figure 2: Bubble chart — top genus × metal hits (kitchen-sink ctrl) ──
# x = beta (effect size), y = genus, colour = metal, size = -log10(pval_fdr)

top_hits = (genus_mwas[genus_mwas['ctrl'] == kitchen]
            .sort_values('pval_fdr')
            .head(60)
            .copy())

if len(top_hits) > 0:
    metals_seen  = sorted(top_hits['metal'].unique())
    genera_seen  = top_hits['genus'].unique()  # ordered by first appearance (ascending pval_fdr)
    metal_to_col = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(metals_seen)}

    fig, ax = plt.subplots(figsize=(FIGW['2col'], max(ROW_H * 1.5, len(genera_seen) * 0.25)))
    for _, row in top_hits.iterrows():
        size  = min(-np.log10(row['pval_fdr'] + 1e-10) * 15, 200)
        color = metal_to_col.get(row['metal'], PALETTE[0])
        ypos  = list(genera_seen).index(row['genus'])
        ax.scatter(row['beta'], ypos, s=size, color=color, alpha=0.8,
                   edgecolors='k', linewidths=0.3)

    ax.set_yticks(range(len(genera_seen)))
    ax.set_yticklabels(genera_seen, fontsize=7)
    ax.axvline(0, color='gray', lw=0.8, ls='--')
    grid_h(ax)
    ax.set_xlabel('β (CLR genus ~ log1p metal)')
    ax.set_ylabel('Genus')
    ax.set_title(f'Top 60 hits — {kitchen}')

    # Legend: metals
    handles = [plt.scatter([], [], s=50, color=c, label=m, edgecolors='k', linewidths=0.3)
               for m, c in list(metal_to_col.items())[:8]]
    ax.legend(handles=handles, title='Element', fontsize=7, title_fontsize=7,
              loc='lower right', ncol=2)

    fig.suptitle('NB34 — Top genus × metal associations (CLR, kitchen-sink control)', y=1.02)
    save(fig, FIGS / 'nb34_genus_mwas_bubble')
    print('Figure 2 saved.')
else:
    print('No hits to plot.')

Figure 2 saved.


In [12]:
# ── Figure 3: Triangulation — compare NB34 genus MWAS with NB33 PGLS ─────
# For each (KO, metal), count how many significant genera in NB34 carry that KO
# and compare to PGLS beta from NB33.

NB33_PGLS = DATA / 'nb33_pgls_abund_weighted.csv'
if NB33_PGLS.exists():
    pgls = pd.read_csv(NB33_PGLS)
    pgls = pgls[pgls['ctrl'] == kitchen.replace('pH+SOC+WTD+redox+metalPC1', 'pH+SOC+WTD+redox+metalPC1')]
    if len(pgls) == 0:
        pgls = pd.read_csv(NB33_PGLS)
        # use closest available ctrl
        available_ctrls = pgls['ctrl'].unique()
        print(f'PGLS controls available: {available_ctrls}')
        pgls = pgls[pgls['ctrl'] == pgls['ctrl'].value_counts().index[0]]

    # NB34 side: for each (ko, metal), count sig genera
    nb34_ko_counts = (sig_ko_strict
        .groupby(['ko', 'metal'])
        .agg(n_sig_genera=('genus', 'nunique'))
        .reset_index())

    merged = nb34_ko_counts.merge(
        pgls[['ko', 'metal', 'beta', 'p_fdr', 'sig_fdr']].rename(columns={'beta': 'beta_pgls'}),
        on=['ko', 'metal'], how='inner'
    )
    print(f'KO × metal pairs in both NB34 and PGLS: {len(merged):,}')

    if len(merged) > 5:
        fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))
        ns_mask = ~merged['sig_fdr'].fillna(False)
        sig_mask = merged['sig_fdr'].fillna(False)
        ax.scatter(merged.loc[ns_mask,  'n_sig_genera'], merged.loc[ns_mask,  'beta_pgls'],
                   s=20, alpha=0.5, color=PALETTE[0], linewidths=0, label='PGLS NS')
        ax.scatter(merged.loc[sig_mask, 'n_sig_genera'], merged.loc[sig_mask, 'beta_pgls'],
                   s=30, alpha=0.9, color='firebrick', linewidths=0, label='PGLS sig')
        ax.axhline(0, color='gray', lw=0.8, ls='--')
        grid_h(ax)
        ax.set_xlabel('N significant genera (NB34 genus MWAS)')
        ax.set_ylabel('PGLS β (KO density ~ metal, NB33)')
        ax.set_title('NB34 × NB33 triangulation')
        ax.legend(fontsize=7)
        fig.suptitle('NB34 — Genus MWAS count vs PGLS β by KO × metal', y=1.02)
        save(fig, FIGS / 'nb34_vs_pgls_triangulation')
        print('Figure 3 saved.')
    else:
        print('Insufficient overlap for triangulation figure.')
else:
    print(f'NB33 PGLS file not found ({NB33_PGLS.name}) — skipping triangulation.')
    print('Re-run this cell after NB33 PGLS completes.')

KO × metal pairs in both NB34 and PGLS: 546
Figure 3 saved.


In [13]:
# ── Final summary ────────────────────────────────────────────────────────
print('=== NB34 complete ===')
print(f'  Genera tested:              {len(KEEP_GENERA):,}')
print(f'  Metals tested:              {len(METAL_COLS)}')
print(f'  Control combos:             {len(CTRL_COMBOS_FINAL)}')
print(f'  Total OLS fits:             {len(genus_mwas):,}')
print(f'  FDR-significant hits:       {int(genus_mwas["sig_fdr"].sum()):,}')
print(f'  Unique genera significant:  {genus_mwas[genus_mwas["sig_fdr"]]["genus"].nunique():,}')
print(f'  Unique metals significant:  {genus_mwas[genus_mwas["sig_fdr"]]["metal"].nunique():,}')
print()
print('Output files:')
print(f'  {OUT_PATH.name}')
print(f'  {OUT_KO_PATH.name}')
print(f'  nb34_ko_metal_summary.csv')
print()
print('Figures:')
print(f'  nb34_genus_mwas_heatmap.pdf')
print(f'  nb34_genus_mwas_bubble.pdf')
print(f'  nb34_vs_pgls_triangulation.pdf')

=== NB34 complete ===
  Genera tested:              200
  Metals tested:              48
  Control combos:             9
  Total OLS fits:             76,107
  FDR-significant hits:       35,987
  Unique genera significant:  200
  Unique metals significant:  48

Output files:
  nb34_genus_mwas_results.csv
  nb34_genus_mwas_ko_annotated.csv
  nb34_ko_metal_summary.csv

Figures:
  nb34_genus_mwas_heatmap.pdf
  nb34_genus_mwas_bubble.pdf
  nb34_vs_pgls_triangulation.pdf
